# LATS: Language Agent Tree Search

Welcome to this notebook on **LATS (Language Agent Tree Search)**. LATS is the natural next step after Tree-of-Thoughts (ToT): it keeps ToT's core idea -- generate multiple candidate reasoning branches with an LLM and search over them -- but replaces ToT's "expand everything, then prune the invalid ones" strategy with the machinery of **Monte Carlo Tree Search (MCTS)**: a learned value/reward signal, a UCB1-style selection rule, and backpropagation of rewards up the tree.

This notebook is a simplified, educational reimplementation of the idea behind the `22_lats` example in FareedKhan-dev's `all-agentic-architectures` collection (the code below is written from scratch, not copied from that repo). We'll apply it to the classic **Game of 24** puzzle: given four numbers, combine them with `+ - * /`, using each number exactly once, to reach the value 24. This is a great fit for search-with-lookahead because a move that looks fine in isolation can dead-end a few steps later, and a bounded budget of LLM calls should be spent on the most promising branches rather than spread evenly across all of them.


### LATS vs. Tree-of-Thoughts (`04_Tree_of_Thoughts.ipynb` in this same folder)

The Tree-of-Thoughts notebook next door already builds a tree of LLM-generated branches: at every step it asks the LLM/environment for *all* valid next moves, keeps every non-cyclical branch, and simply repeats until one branch reaches the goal. That works, but the tree grows breadth-first and unboundedly -- by the last round of the wolf/goat/cabbage puzzle, ToT was tracking 32 simultaneous paths.

**LATS keeps ToT's branching idea but adds three MCTS mechanics on top of it:**

1. **A value/reward model.** Instead of only checking "is this branch valid or not", an LLM-as-judge scores *how promising* a candidate partial solution is (e.g. 0-10). This is a continuous signal, not a binary keep/prune decision.
2. **UCB1-style selection.** Rather than expanding every branch every round, LATS uses the classic UCB1 formula (`average_value + exploration_bonus`) to decide *which single node* to expand next -- balancing exploiting branches that have scored well so far against exploring branches that haven't been tried much yet.
3. **Backpropagation.** Once a new candidate is scored, that reward is propagated back up through every ancestor on the path to the root, updating their visit counts and cumulative value. This lets information discovered deep in the tree influence selection decisions near the root on the *next* iteration.

In short: **LATS = ToT's "LLM proposes branches" + MCTS's select -> expand -> evaluate -> backpropagate loop.** This makes LATS more sample-efficient on problems with large branching factors, at the cost of needing a working value/reward model and some extra bookkeeping.


### Definition
**Language Agent Tree Search (LATS)** is an agentic reasoning framework that frames problem solving as Monte Carlo Tree Search over a tree of LLM-generated reasoning steps. Each node holds a partial solution. An LLM proposes children from a node (**expansion**), an LLM-as-judge estimates how promising each child is (**evaluation**, standing in for a full MCTS rollout/simulation), a UCB1 formula picks which node to expand next (**selection**), and the observed reward is pushed back up the path to the root so every ancestor's statistics improve (**backpropagation**). Repeating this loop for a bounded number of iterations converges the search toward a high-value root-to-leaf path, which becomes the final answer.

### High-level Workflow

1.  **Selection:** Starting at the root, descend the tree by repeatedly picking the child with the best UCB1 score, until you reach a node that has no children yet (a search "frontier").
2.  **Expansion:** Ask the LLM to propose K candidate next steps/actions from that frontier node's state, creating new child nodes.
3.  **Evaluation:** Score each new child's promise with an LLM-as-judge (0-10). This is the reward signal that stands in for a full rollout-to-the-end simulation.
4.  **Backpropagation:** Update `visits` and cumulative `value` for the newly scored node and every ancestor back up to the root.
5.  **Repeat** steps 1-4 for a fixed number of iterations (a search budget).
6.  **Extraction:** Walk the tree from the root, always following the child with the best average value, to read off the final root-to-leaf answer path.

### When to Use / Applications
*   **Planning/puzzle problems with a large branching factor**, where fully expanding every branch like ToT does becomes too expensive, but a reasonable value estimate for a partial solution is available.
*   **Tasks with a usable reward signal** -- an LLM-as-judge, a unit test, a verifier function, or partial-credit heuristic -- since the quality of the search is only as good as the quality of this reward.
*   **Code generation and multi-step agent tool-use**, where intermediate states can be scored for progress and a bounded extra budget of LLM calls buys meaningfully better multi-step solutions than one-shot generation.

### Strengths & Weaknesses
*   **Strengths:**
    *   **Sample-efficient search:** UCB1 selection focuses the LLM-call budget on promising subtrees instead of expanding every branch every round like ToT.
    *   **Continuous value signal:** A 0-10 promise score is much richer than ToT's binary valid/invalid pruning.
    *   **Information sharing via backpropagation:** Rewards discovered deep in the tree sharpen decisions made near the root on later iterations.
*   **Weaknesses:**
    *   **More moving parts:** Needs a value/reward model, an exploration constant, and tree bookkeeping on top of what ToT already needs.
    *   **Noisy reward signal:** An LLM-as-judge score is an approximation, not ground truth, and can misguide the search.
    *   **Still expensive:** More LLM calls than a single direct prompt, and for small problems, plain ToT (or even Chain-of-Thought) may be "good enough".


## Phase 0: Foundation & Setup

We use the repo's shared `helpers` factory (`get_llm`) so this notebook stays platform-aware (Groq on Windows, Databricks on macOS) rather than hardcoding a specific chat model client, per this repo's conventions for LangGraph-phase / agent-pattern notebooks.

In [ ]:
import math
import random
from dataclasses import dataclass, field
from typing import List, Optional

from pydantic import BaseModel, Field

from helpers import get_llm

llm = get_llm()
print("LLM initialized.")


## Phase 1: The `Node` class -- the search tree's building block

Every node in the LATS tree represents a **partial solution** to the Game of 24 puzzle: the list of numbers still "in play", plus the sequence of operations that got us there. Alongside the state we track the MCTS bookkeeping every node needs: a link to its `parent`, its `children`, how many times it has been `visits`-ed by the search, and the cumulative `value` (sum of rewards) backpropagated through it. `average_value()` (value / visits) is the number UCB1 and the final path-extraction step actually care about.

In [ ]:
@dataclass
class Node:
    """A node in the LATS search tree: a partial solution to the Game of 24."""
    numbers: List[float]                       # numbers still available to combine
    history: List[str] = field(default_factory=list)  # operations applied so far, root to here
    parent: Optional["Node"] = None
    children: List["Node"] = field(default_factory=list)
    visits: int = 0
    value: float = 0.0                          # cumulative backpropagated reward

    def is_terminal(self) -> bool:
        """A terminal state has exactly one number left -- nothing more to combine."""
        return len(self.numbers) == 1

    def is_solved(self) -> bool:
        return self.is_terminal() and abs(self.numbers[0] - 24) < 1e-6

    def average_value(self) -> float:
        return self.value / self.visits if self.visits > 0 else 0.0

    def state_text(self) -> str:
        hist = " -> ".join(self.history) if self.history else "(start)"
        return f"Remaining numbers: {self.numbers} | Steps so far: {hist}"


TARGET = 24
root = Node(numbers=[4, 7, 8, 8])
print(root.state_text())


## Phase 2: `expand` -- LLM-generated candidate next steps

`expand(node)` is the "Tree-of-Thoughts" half of LATS: we ask the LLM to propose `K` different single arithmetic operations (combine exactly two of the currently available numbers with `+ - * /`) that could plausibly still lead to 24. We use structured output so each candidate reliably gives us both a human-readable `operation` string and the resulting `numbers` list, which we use to build the new child `Node`s.

In [ ]:
class Candidate(BaseModel):
    """One candidate next step: combine two of the current numbers."""
    operation: str = Field(description="Human-readable operation, e.g. '8 - 4 = 4'")
    resulting_numbers: List[float] = Field(
        description="The full list of numbers remaining after applying the operation "
        "(the two combined numbers replaced by their result; all other numbers unchanged)."
    )


class CandidateList(BaseModel):
    candidates: List[Candidate] = Field(description="K distinct candidate next steps.")


def expand(node: "Node", k: int = 3) -> List["Node"]:
    """Ask the LLM for K candidate next steps from `node`, and materialize them as child Nodes."""
    structured_llm = llm.with_structured_output(CandidateList)
    prompt = (
        "You are playing the Game of 24. You must combine numbers with +, -, *, / "
        "(using each number exactly once overall) to reach exactly 24.\n\n"
        f"Current numbers: {node.numbers}\n"
        f"Steps taken so far: {node.history if node.history else '(none)'}\n\n"
        f"Propose {k} different, valid single operations: pick exactly two of the current "
        "numbers and combine them with one of +, -, *, /. Return the operation as text and the "
        "full resulting list of numbers (replace the two used numbers with the result; leave the "
        "others untouched). Prefer operations that plausibly move toward reaching 24."
    )
    result: CandidateList = structured_llm.invoke(prompt)

    children = []
    for cand in result.candidates[:k]:
        child = Node(
            numbers=cand.resulting_numbers,
            history=node.history + [cand.operation],
            parent=node,
        )
        children.append(child)
    node.children = children
    return children


## Phase 3: `evaluate` -- the value/reward model (LLM-as-judge)

This is the piece ToT doesn't have. For a **terminal** node (one number left), the reward is deterministic: 10 if it equals 24, 0 otherwise -- no need to ask an LLM. For a **non-terminal** node, we ask an LLM judge to rate, on a 0-10 scale, how promising this partial state looks as a step toward 24. This score is the "reward" that gets backpropagated up the tree.

In [ ]:
class Score(BaseModel):
    score: int = Field(description="Promise score from 0 (dead end) to 10 (very likely to reach 24).", ge=0, le=10)
    rationale: str = Field(description="One short sentence justifying the score.")


def evaluate(node: "Node") -> float:
    """Return a 0-10 reward estimate for how promising this node's state is."""
    if node.is_terminal():
        return 10.0 if node.is_solved() else 0.0

    judge_llm = llm.with_structured_output(Score)
    prompt = (
        "You are an expert judge for the Game of 24. Given the remaining numbers below, rate "
        "from 0 to 10 how likely it is that some sequence of +, -, *, / operations on these "
        "numbers (using each exactly once) can still reach exactly 24. 10 = almost certainly "
        "solvable from here, 0 = clearly a dead end.\n\n"
        f"Remaining numbers: {node.numbers}\n"
        f"Steps so far: {node.history}"
    )
    result: Score = judge_llm.invoke(prompt)
    return float(result.score)


## Phase 4: `select` -- UCB1-style tree policy

Given a node with children that already have visit/value statistics, UCB1 scores each child as:

$$\text{UCB1}(child) = \underbrace{\frac{child.value}{child.visits}}_{\text{exploitation}} + \underbrace{c \cdot \sqrt{\frac{\ln(parent.visits)}{child.visits}}}_{\text{exploration bonus}}$$

The first term favors children that have scored well on average so far (**exploitation**); the second term grows for children that haven't been visited much relative to their parent (**exploration**), so the search doesn't get stuck always re-expanding the single best-looking branch. An unvisited child (`visits == 0`) is given infinite priority so every child gets tried at least once before UCB1 statistics are trusted.

In [ ]:
EXPLORATION_CONSTANT = 1.4  # classic sqrt(2)-ish UCB1 constant


def ucb1_score(child: "Node", parent_visits: int, c: float = EXPLORATION_CONSTANT) -> float:
    if child.visits == 0:
        return float("inf")  # always try unvisited children first
    exploitation = child.value / child.visits
    exploration = c * math.sqrt(math.log(parent_visits) / child.visits)
    return exploitation + exploration


def select(node: "Node") -> "Node":
    """Descend from `node` via UCB1 until we reach a node with no children (a search frontier)."""
    current = node
    while current.children and not current.is_terminal():
        current = max(current.children, key=lambda ch: ucb1_score(ch, max(current.visits, 1)))
    return current


## Phase 5: `backpropagate` -- pushing rewards back up the tree

After a newly expanded node is evaluated, `backpropagate` walks from that node back up through every `parent` link to the root, incrementing `visits` and adding the observed `reward` to `value` at every level. This is what lets a good (or bad) discovery deep in the tree influence the UCB1 scores that `select` will see at the *top* of the tree on the next iteration.

In [ ]:
def backpropagate(node: "Node", reward: float) -> None:
    current: Optional[Node] = node
    while current is not None:
        current.visits += 1
        current.value += reward
        current = current.parent


## Phase 6: The search loop

Each iteration of LATS does one full **select -> expand -> evaluate -> backpropagate** cycle:

1.  `select(root)` walks down via UCB1 to a frontier node (no children yet).
2.  If that frontier node is terminal, we just evaluate and backpropagate it directly (nothing to expand).
3.  Otherwise we `expand` it into K children, `evaluate` every new child, and `backpropagate` each child's reward.

We run this for a small, bounded number of iterations (a real system might run hundreds; here we keep it small and print the tree after every step so the search process stays visible).

**What we are going to do:**
Run a bounded number of LATS iterations (select -> expand -> evaluate -> backpropagate) on the Game of 24 puzzle `[4, 7, 8, 8]`, printing the tree after each iteration so the search process is visible.

In [ ]:
def print_tree(node: "Node", indent: str = "") -> None:
    marker = " (SOLVED)" if node.is_solved() else ""
    print(f"{indent}- {node.state_text()} | visits={node.visits} avg_value={node.average_value():.2f}{marker}")
    for child in node.children:
        print_tree(child, indent + "    ")


def run_lats_iteration(root: "Node", k: int = 3) -> None:
    frontier = select(root)

    if frontier.is_terminal():
        reward = evaluate(frontier)
        backpropagate(frontier, reward)
        return

    children = expand(frontier, k=k)
    for child in children:
        reward = evaluate(child)
        backpropagate(child, reward)


In [ ]:
N_ITERATIONS = 8
K_CANDIDATES = 3

random.seed(7)  # only affects any tie-breaking we might add; LLM calls are the real source of variation

for i in range(1, N_ITERATIONS + 1):
    print(f"=== LATS iteration {i}/{N_ITERATIONS} ===")
    run_lats_iteration(root, k=K_CANDIDATES)
    print_tree(root)
    print()


**Discussion of the Output:**
Watch how the tree grows unevenly across iterations -- that's the point. Early iterations spend their budget trying an unvisited child at every branch point (UCB1 gives `visits == 0` children infinite priority). Once every child at a level has been tried once, UCB1 starts favoring the branch with the best `avg_value` so far, occasionally still poking at a less-visited sibling when its exploration bonus grows large enough. This is exactly the exploitation/exploration trade-off that plain ToT's "expand every branch, every round" strategy does not make -- ToT would have kept all of these branches alive simultaneously regardless of how promising the judge thought they were.

## Phase 7: Extracting the final answer -- walking the best-value path

Once the search budget is spent, we don't need to inspect the whole tree by hand: we walk from the root, always following the child with the highest `average_value()`, until we hit a leaf. That root-to-leaf path *is* the LATS agent's final answer -- the sequence of operations it believes is most likely to reach (or get closest to) 24.

In [ ]:
def extract_best_path(root: "Node") -> List["Node"]:
    path = [root]
    current = root
    while current.children:
        current = max(current.children, key=lambda c: c.average_value())
        path.append(current)
    return path


best_path = extract_best_path(root)

print("--- Best path found by LATS ---")
for step_num, node in enumerate(best_path):
    print(f"{step_num}. {node.state_text()}  (visits={node.visits}, avg_value={node.average_value():.2f})")

final_node = best_path[-1]
if final_node.is_solved():
    print("\nSolution found! Final expression reaches 24.")
else:
    print(f"\nBest attempt did not reach exactly 24 within the search budget; "
          f"ended at {final_node.numbers} after {N_ITERATIONS} iterations. "
          "Increase N_ITERATIONS / K_CANDIDATES for a better chance of finding an exact solution.")


**Discussion of the Output:**
Because our search budget is intentionally tiny (a handful of iterations, a handful of candidates per expansion) this may or may not land on an exact solution to `[4, 7, 8, 8] -> 24` -- a real classic answer being e.g. `(4 - 8/8) * 7 = 24` (or several structurally similar ones). What matters for this notebook is *how* the answer was produced: not a single LLM guess, but the path through the tree with the best average reward across all the times it was visited and re-evaluated, discovered by balancing exploration of new branches against exploitation of the ones the judge liked. Raising `N_ITERATIONS` and/or `K_CANDIDATES` -- trading more LLM calls for a wider, deeper search -- is the direct lever for improving solution quality, exactly as it is in real MCTS-based systems.

## Conclusion

In this notebook we implemented **LATS (Language Agent Tree Search)**: a Tree-of-Thoughts-style tree of LLM-generated branches, upgraded with the three core mechanics of Monte Carlo Tree Search:

*   an **LLM-as-judge value/reward model** (`evaluate`) that scores how promising a partial solution is, not just whether it's valid,
*   a **UCB1-style selection rule** (`select`) that spends the search budget on the most promising *and* under-explored branches instead of expanding every branch every round like plain ToT, and
*   **backpropagation** (`backpropagate`) that pushes newly observed rewards back up through every ancestor, so information discovered deep in the tree sharpens decisions made near the root on later iterations.

This combination makes LATS more sample-efficient than brute-force ToT on problems with large branching factors, at the cost of needing a working value/reward signal and some extra bookkeeping (`Node.visits`, `Node.value`, the exploration constant).

**Where this sits in the roadmap:** LATS belongs in **Layer B4 -- "Sampling and search"**, alongside **Self-Consistency**, **Tree of Thoughts**, **Ensemble**, and **Mental Loop** -- the family of patterns that trade extra inference-time compute (more LLM calls, more candidates, more search) for higher-quality, more reliable answers than a single direct generation.